# ML-10 - Content Action Playbook

This notebook turns the capstone scores into actions a human reviewer could use.

## 1. Ranked actions + reason codes

The final queue blends model probability with the Week-4 baseline score. The reason codes explain why a row is present. The top recommendations in this run mostly surface deep, zero-click pages at high modeled decline risk, so the action is usually `monitor_or_prune`, not automatic refresh.

In [1]:

from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown, Image


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "work" / "scripts" / "run_capstone_analysis.py").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root")

ROOT = find_repo_root(Path.cwd().resolve())
METRICS_PATH = ROOT / "work" / "outputs" / "capstone_metrics.json"
if not METRICS_PATH.exists():
    raise FileNotFoundError(
        "Missing work/outputs/capstone_metrics.json. Run `python work/scripts/run_capstone_analysis.py` from the repo root after Hugging Face login."
    )
metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
results = pd.DataFrame(metrics["results"])
top_features = pd.DataFrame(metrics["top_features"])
top20 = pd.DataFrame(metrics["top20_preview"])
print(f"Loaded metrics from {METRICS_PATH.relative_to(ROOT)}")
print(f"Eligible rows: {metrics['eligible_rows']:,}; clients: {metrics['eligible_clients']}; base rate: {metrics['base_rate']:.3f}")

cols = ['rank', 'content_hash_id', 'final_action_score', 'action_label', 'reason_codes', 'impressions_may', 'ctr_may', 'avg_position_may', 'future_decline_label']
display(top20[cols].head(20))

Loaded metrics from work\outputs\capstone_metrics.json
Eligible rows: 24,547; clients: 18; base rate: 0.256


,rank,content_hash_id,final_action_score,action_label,reason_codes,impressions_may,ctr_may,avg_position_may,future_decline_label
0,1,content_e5119f5ccb21b8be,69.31,monitor_or_prune,future_decline_risk;deep_zero_click_risk,256,0.0,80.734375,1
1,2,content_84332ce4be404e3f,69.27,monitor_or_prune,future_decline_risk;deep_zero_click_risk,306,0.0,76.669935,1
2,3,content_e02ca086a84cfd35,69.14,monitor_or_prune,future_decline_risk;deep_zero_click_risk,228,0.0,81.000000,1
3,4,content_ad0b68d1727295e3,69.11,monitor_or_prune,future_decline_risk;deep_zero_click_risk,304,0.0,82.569079,1
4,5,content_905103df18204769,69.10,monitor_or_prune,future_decline_risk;deep_zero_click_risk,263,0.0,75.041825,1
5,6,content_dcdb257bd5040270,69.09,monitor_or_prune,future_decline_risk;deep_zero_click_risk,241,0.0,84.514523,1
6,7,content_305afb885f6a3b9c,69.08,monitor_or_prune,future_decline_risk;deep_zero_click_risk,537,0.0,78.370577,1
7,8,content_df405a5ed66aab92,69.07,monitor_or_prune,future_decline_risk;deep_zero_click_risk,308,0.0,75.564935,1
8,9,content_7faa8052b9d6529d,69.06,monitor_or_prune,future_decline_risk;deep_zero_click_risk,375,0.0,84.600000,1
9,10,content_80be8e495d9ac06a,69.04,monitor_or_prune,future_decline_risk;deep_zero_click_risk,418,0.0,79.440191,1


## 2. Intended use and limits

Intended user: a content strategist or SEO specialist reviewing candidates in priority order.

Use: inspect pages with high scores, check query intent and SERP context, then choose whether to monitor, prune, consolidate, refresh title/meta, or review engagement.

Limit: the score is not an instruction to edit. It is a prioritization tool built from observed warehouse metrics.

In [2]:
action_counts = top20['action_label'].value_counts().rename_axis('action_label').reset_index(name='top20_count')
display(action_counts)

,action_label,top20_count
0,monitor_or_prune,20


## 3. Human review + the no-go list

Before action, a person must check whether the page is strategically important, duplicated by another page, seasonal, blocked by tracking gaps, or intentionally low-click.

No-go list: do not publish client names, domains, URLs, raw queries, credentials, or claims that a refresh caused recovery.

In [3]:
review_checks = pd.DataFrame([
    {"check": "Demand", "question": "Does this page have enough impressions or business value to justify work?"},
    {"check": "Intent", "question": "Could query intent or SERP layout explain the low CTR or decline?"},
    {"check": "Consolidation", "question": "Is another related page absorbing the demand?"},
    {"check": "Tracking", "question": "Are GSC/GA4 coverage gaps making the page look worse than it is?"},
])
display(review_checks)

,check,question
0,Demand,Does this page have enough impressions or busi...
1,Intent,Could query intent or SERP layout explain the ...
2,Consolidation,Is another related page absorbing the demand?
3,Tracking,Are GSC/GA4 coverage gaps making the page look...


## 4. Monitoring / retrain triggers

Retrain or re-audit when the month changes, the base rate shifts, a client loses or gains analytics access, the top recommendations become mostly weak picks, or Precision@K drops below the baseline on a fresh holdout.

In [4]:
triggers = pd.DataFrame([
    {"trigger": "Monthly refresh", "response": "Rebuild feature and target windows."},
    {"trigger": "Tracking access change", "response": "Recheck GA4/GSC availability filters."},
    {"trigger": "Metric drift", "response": "Compare base rate and Precision@K against this run."},
    {"trigger": "Weak top queue", "response": "Review reason codes and adjust action labels before modeling."},
])
display(triggers)

,trigger,response
0,Monthly refresh,Rebuild feature and target windows.
1,Tracking access change,Recheck GA4/GSC availability filters.
2,Metric drift,Compare base rate and Precision@K against this...
3,Weak top queue,Review reason codes and adjust action labels b...


## 5. Exports for the paper

The analysis writes the paper inputs under `work/outputs/` and `work/figures/`. CSV queues are ignored by git; JSON receipts and figures are committed.

In [5]:
display(pd.DataFrame([metrics['outputs']]).T.rename(columns={0: 'path'}))

,path
feature_table_csv,work/outputs/capstone_feature_table.csv
ranked_recommendations_csv,work/outputs/capstone_ranked_recommendations.csv
top20_csv,work/outputs/capstone_top20_recommendations.csv
model_vs_baseline_chart,work/figures/capstone_model_vs_baseline.png
feature_importance_chart,work/figures/capstone_feature_importance.png
action_mix_chart,work/figures/capstone_action_mix.png


## Self-check

- [x] Ranked actions and reason codes are shown.
- [x] Intended use and limits are named.
- [x] Human review checks and no-go list are included.
- [x] Paper exports are documented.